# B2 · 테스트셋 1회 검증 + 원눈금 채점표 (7문항 최종확정판)

팀 최종 결정: **7문항(고정6 + Q7전화)**. 8문항(+Q23가전사용)은 TEST에서 중증놓침 22.2%로 A0(20.8%)를
못 이겨 탈락, 7문항이 MAE(0.538)·중증놓침(19.4% ≤ A0 20.8%) 둘 다 확정 통과.

원본은 `sjlee`님 로컬 스크립트(`b2_test_and_scoretable.py`, Windows 로컬 경로 + `_recon_from_raw`
모듈 사용)를 그대로 옮긴 게 아니라, **Google Drive 마운트 방식(우리 표준 워크플로우)으로 데이터 로드
부분만 재구성**한 버전입니다. 로직(τ 튜닝, 후보 비교, 원눈금 채점표 산출)은 동일합니다.

⚠️ `_recon_from_raw` 모듈이 정확히 어떤 후처리를 하는지 확인 못 했으므로, 아래 1번 섹션에서
`adl_wide.csv`(VISITNUM==2.0) + `baseline_sample.csv`를 직접 병합하는 방식으로 대체했습니다.
실행 후 `dev=1,652 test=551`이 그대로 재현되는지 꼭 확인하세요(재현 안 되면 원본 병합 로직과
다른 부분이 있다는 뜻이니 sjlee님께 `_recon_from_raw.py` 내용 문의 필요).

## 0. 설정 — Drive 마운트 및 경로

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score
warnings.filterwarnings("ignore")

DATA_DIR = Path("/content/drive/MyDrive/2026 urp/preprocessed")
OUT_DIR = Path("/content/drive/MyDrive/2026 urp/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_PATH = DATA_DIR / "baseline_sample.csv"
ADL_MAIN_PATH = DATA_DIR / "adl_wide.csv"

print("경로 확인:")
for p in [BASELINE_PATH, ADL_MAIN_PATH]:
    print(f"  {'✅' if p.exists() else '❌ 없음'}  {p}")


## 1. 상수·후보 정의 (원본과 동일, 팀 확정 반영)

In [ ]:
TRIALS = ["AD-1061", "AD-1063", "AD-1064"]
STAGES = np.arange(6)
GRID = np.round(np.arange(0.02, 0.60, 0.02), 2)

LAB = {"ADL0101":"Q1먹기","ADL0103":"Q3화장실","ADL0104":"Q4목욕","ADL0105":"Q5몸단장",
       "ADL0106A":"Q6a옷고르기","ADL0106B":"Q6b옷입기","ADL0107A":"Q7전화","ADL0112A":"Q12집안일",
       "ADL0115A":"Q15외출","ADL0116A":"Q16a쇼핑","ADL0117A":"Q17금전","ADL0122Q":"Q22취미",
       "ADL0123L":"Q23가전사용"}
FIXED6 = ["ADL0103", "ADL0104", "ADL0105", "ADL0106A", "ADL0115A", "ADL0116A"]

# ⭐ 팀 최종 확정 = 7문항(고정6 + Q7전화). Q17금전은 Q7전화와 같은 요인축이라 파시모니로 제외.
PRIMARY = "7문항 확정(+Q7전화)"

CAND = {  # 추가검증 페이지 후보 전부 (7문항×2, 8문항×3, 9문항×3) — 비교용으로 유지
    "7문항 주(+Q17금전)":            FIXED6 + ["ADL0117A"],
    "7문항 확정(+Q7전화)":           FIXED6 + ["ADL0107A"],
    "8문항 주(+Q7전화,Q17금전)":     FIXED6 + ["ADL0107A", "ADL0117A"],
    "8문항 보조1(+Q1먹기,Q7전화)":   FIXED6 + ["ADL0101", "ADL0107A"],
    "8문항 보조2(+Q1먹기,Q17금전)":  FIXED6 + ["ADL0101", "ADL0117A"],
    "9문항 주(+Q6b,Q17,Q23)":       FIXED6 + ["ADL0106B", "ADL0117A", "ADL0123L"],
    "9문항 보조1(+Q6b,Q12,Q17)":    FIXED6 + ["ADL0106B", "ADL0112A", "ADL0117A"],
    "9문항 보조2(+Q1,Q6b,Q22)":     FIXED6 + ["ADL0101", "ADL0106B", "ADL0122Q"],
}

ALL_ITEMS_NEEDED = sorted(set(it for cols in CAND.values() for it in cols))
print("필요 문항 전체:", ALL_ITEMS_NEEDED)


## 2. 헬퍼 함수 (원본과 동일)

In [ ]:
def mae(t, p): return np.abs(np.asarray(t, float) - np.asarray(p, float)).mean()

def miss(t, p):
    t = np.asarray(t, float); p = np.asarray(p, float); hi = t >= 4
    return (p[hi] <= 3).mean() if hi.sum() else np.nan

def exact(t, p): return (np.asarray(t, int) == np.asarray(p, int)).mean()
def within1(t, p): return (np.abs(np.asarray(t, float) - np.asarray(p, float)) <= 1).mean()
def cmed(P): return STAGES[(np.cumsum(P, 1) >= 0.5).argmax(1)]

def asym(P, t4, t5):
    p5 = P[:, 5]; p4 = P[:, 4] + P[:, 5]
    return np.where(p5 > t5, 5, np.where(p4 > t4, 4, cmed(P)))

def fit_std(Xtr, ytr):
    sc = StandardScaler().fit(Xtr); Z = sc.transform(Xtr)
    md_ = {k: LogisticRegression(penalty="l2", solver="lbfgs", C=1.0, max_iter=5000).fit(Z, (ytr >= k).astype(int))
           for k in range(1, 6)}
    return sc, md_

def P_(sc, md_, X):
    Z = sc.transform(X); g = {k: md_[k].predict_proba(Z)[:, 1] for k in range(1, 6)}
    P = np.zeros((len(X), 6)); P[:, 0] = 1 - g[1]
    for k in range(1, 5): P[:, k] = g[k] - g[k + 1]
    P[:, 5] = g[5]; P = np.clip(P, 1e-9, None); P /= P.sum(1, keepdims=True)
    return P

def tune(Ptr, ytr, budget):
    best = None
    for t4 in GRID:
        for t5 in GRID:
            if t5 < t4: continue
            p = asym(Ptr, t4, t5)
            if miss(ytr, p) <= budget + 1e-9:
                key = mae(ytr, p)
                if best is None or key < best[0]: best = (key, t4, t5)
    if best is None:
        c = [(miss(ytr, asym(Ptr, a, b)), mae(ytr, asym(Ptr, a, b)), a, b) for a in GRID for b in GRID if b >= a]
        _, _, a, b = min(c); return a, b
    return best[1], best[2]


## 3. 데이터 로드 + 공식 재현 분할

`_recon_from_raw` 대체: `adl_wide.csv`(VISITNUM==2.0, 문항점수) + `baseline_sample.csv`
(ds_stage/A0_harmonized/A1_2015_stage, `in_common_comparison_sample==True` 필터) 직접 병합.

In [ ]:
bs = pd.read_csv(BASELINE_PATH)
adl_main = pd.read_csv(ADL_MAIN_PATH)

# baseline_sample.csv: in_common_comparison_sample==True만 (원본 com 재현)
com = bs[bs["in_common_comparison_sample"] == True][
    ["STUDYID", "USUBJID", "ds_stage", "A0_harmonized", "A1_2015_stage"]].copy()
com["ds_stage"] = com["ds_stage"].astype(int)

# ⭐ 수정: IADL 문항은 관문(gate) 로직 때문에 원본 컬럼(ADL0107A 등)에 스킵으로 인한 NaN이
#   남아있음. '__resolved' 컬럼이 있으면 그걸 우선 사용(관문 스킵 처리 완료본).
base_visit = adl_main[adl_main["VISITNUM"] == 2.0].copy()

def resolve_col(item):
    resolved_name = f"{item}__resolved"
    return resolved_name if resolved_name in base_visit.columns else item

source_map = {it: resolve_col(it) for it in ALL_ITEMS_NEEDED}
print("문항별 실제 사용 컬럼 (원본 vs __resolved):")
for it, src in source_map.items():
    tag = "__resolved 사용" if src != it else "원본 그대로"
    print(f"  {it}({LAB.get(it,'?')}): {src}  [{tag}]")

item_cols = base_visit[["STUDYID", "USUBJID"] + list(source_map.values())].copy()
item_cols = item_cols.rename(columns={v: k for k, v in source_map.items() if v != k})
item_cols = item_cols[["STUDYID", "USUBJID"] + ALL_ITEMS_NEEDED]  # 컬럼 순서/이름 정리

full = com.merge(item_cols, on=["STUDYID", "USUBJID"], how="left")

# 공식 분할 레시피 재현: STUDYID×ds_stage 층화, test_size=0.25, seed=20260814
strata = full["STUDYID"].astype(str) + "_" + full["ds_stage"].astype(str)
_, test_idx = train_test_split(full.index, test_size=0.25, stratify=strata, random_state=20260814)
full["split"] = np.where(full.index.isin(test_idx), "test", "dev")

dev = full[full.split == "dev"].reset_index(drop=True)
test = full[full.split == "test"].reset_index(drop=True)

print(f"\ndev={len(dev)}  test={len(test)}")
print("\n문항별 결측 인원 (수정 후 — __resolved 반영):")
for it in ALL_ITEMS_NEEDED:
    print(f"  {it}({LAB.get(it,'?')}): 결측 {full[it].isna().sum()}명")

print(f"\ndev A0 놓침(예산)={miss(dev.ds_stage, dev.A0_harmonized)*100:.2f}%  "
      f"test A0 놓침={miss(test.ds_stage, test.A0_harmonized)*100:.2f}%")
print(f"test 전체 A1: MAE={mae(test.ds_stage, test.A1_2015_stage):.3f} "
      f"놓침={miss(test.ds_stage, test.A1_2015_stage)*100:.2f}% "
      f"카파={cohen_kappa_score(test.ds_stage, test.A1_2015_stage.astype(int), weights='quadratic'):.3f}")
print(f"test 전체 A0: MAE={mae(test.ds_stage, test.A0_harmonized):.3f} "
      f"놓침={miss(test.ds_stage, test.A0_harmonized)*100:.2f}% "
      f"카파={cohen_kappa_score(test.ds_stage, test.A0_harmonized.astype(int), weights='quadratic'):.3f}")


## 4. TEST 1회 검증 — dev 전체 적합+τ튜닝 → test 예측

8개 후보 전부 한 번씩(비교용 유지), ★ = 팀 최종 확정(7문항+Q7전화).

In [ ]:
print("=" * 94)
print("TEST 1회 검증 — dev 전체 적합+τ튜닝 → test 예측 (★=최종 확정)")
print("=" * 94)
print(f"{'후보':<26}{'n':>5}{'MAE':>7}{'놓침%':>7}{'카파':>7}{'±1%':>7}{'정확%':>7}  {'τ4/τ5':>9}")

results = {}
for name, cols in CAND.items():
    dtr = dev.dropna(subset=cols); dte = test.dropna(subset=cols)
    sc, md_ = fit_std(dtr[cols].values, dtr["ds_stage"].values)
    Ptr = P_(sc, md_, dtr[cols].values)
    budget = miss(dtr["ds_stage"].values, dtr["A0_harmonized"].values)
    t4, t5 = tune(Ptr, dtr["ds_stage"].values, budget)
    Pte = P_(sc, md_, dte[cols].values); pte = asym(Pte, t4, t5)

    r = dict(n=len(dte), mae=mae(dte.ds_stage, pte), miss=miss(dte.ds_stage, pte) * 100,
             kappa=cohen_kappa_score(dte.ds_stage, pte.astype(int), weights="quadratic"),
             w1=within1(dte.ds_stage, pte) * 100, ex=exact(dte.ds_stage, pte) * 100, t4=t4, t5=t5,
             a1_mae=mae(dte.ds_stage, dte.A1_2015_stage), a1_miss=miss(dte.ds_stage, dte.A1_2015_stage) * 100,
             a0_miss=miss(dte.ds_stage, dte.A0_harmonized) * 100)
    results[name] = r
    star = "★" if name == PRIMARY else " "
    print(f"{star}{name:<25}{r['n']:>5}{r['mae']:>7.3f}{r['miss']:>7.2f}{r['kappa']:>7.3f}"
          f"{r['w1']:>7.1f}{r['ex']:>7.1f}  {t4:>4.2f}/{t5:<4.2f}")

p = results[PRIMARY]
print(f"\n── 최종 확정({PRIMARY}) test 판정 ──")
print(f"  vs A1(동일 test행): MAE {p['mae']:.3f} {'<' if p['mae'] < p['a1_mae'] else '≥'} {p['a1_mae']:.3f} | "
      f"놓침 {p['miss']:.1f}% {'≤' if p['miss'] <= p['a1_miss'] else '>'} A1 {p['a1_miss']:.1f}%")
print(f"  vs A0(동일 test행) 놓침: {p['miss']:.1f}% "
      f"{'≤' if p['miss'] <= p['a0_miss'] + 1e-9 else '>'} A0 {p['a0_miss']:.1f}%")


## 5. 원눈금 채점표 — 최종 확정 7문항(고정6+Q7전화), dev+test 재학습

기존 산출 결과(참고용, 이미 확보된 값): `τ4=0.08, τ5=0.56, budget_A0_miss=17.81%, n_train=2196`.
아래 셀 실행 결과가 이 값과 일치하는지 확인하세요.

In [ ]:
print("=" * 94)
print(f"원눈금 채점표 — 최종 확정 7문항(고정6+Q7전화), dev+test 재학습")
print("=" * 94)

cols = CAND[PRIMARY]
allrows = full.dropna(subset=cols).reset_index(drop=True)
X = allrows[cols].values.astype(float); y = allrows["ds_stage"].values
sc = StandardScaler().fit(X); mu = sc.mean_; sd = sc.scale_; Z = sc.transform(X)

rows = []
fitted_models = {}
for k in range(1, 6):
    m = LogisticRegression(penalty="l2", solver="lbfgs", C=1.0, max_iter=5000).fit(Z, (y >= k).astype(int))
    fitted_models[k] = m
    cstd = m.coef_[0]; b_std = m.intercept_[0]
    craw = cstd / sd                                   # 원눈금 계수
    b_raw = b_std - np.sum(cstd * mu / sd)              # 원눈금 절편
    for j, it in enumerate(cols):
        rows.append(dict(cutpoint=f"P(Y>={k})", item_code=it, item_label=LAB[it],
                          ridge_coef_raw=round(craw[j], 4), ridge_coef_standardized=round(cstd[j], 4)))
    rows.append(dict(cutpoint=f"P(Y>={k})", item_code="(intercept)", item_label="(절편)",
                      ridge_coef_raw=round(b_raw, 4), ridge_coef_standardized=round(b_std, 4)))

score = pd.DataFrame(rows)

Pall = P_(sc, fitted_models, X)
budget_all = miss(y, allrows["A0_harmonized"].values)
T4, T5 = tune(Pall, y, budget_all)

score.to_csv(OUT_DIR / "b2_scoretable_7item_raw.csv", index=False, encoding="utf-8-sig")
rule = pd.DataFrame([dict(
    rule="asymmetric_threshold", tau4=T4, tau5=T5, budget_A0_miss=round(budget_all * 100, 2),
    decision="p5=P(Y=5); p4=P(Y>=4); 예측=5 if p5>tau5 elif 4 if p4>tau4 else 조건부중앙값",
    n_train=len(allrows), penalty="L2(C=1.0, standardized)")])
rule.to_csv(OUT_DIR / "b2_scoretable_7item_rule.csv", index=False, encoding="utf-8-sig")

print(score.to_string(index=False))
print(f"\nτ4={T4:.2f}  τ5={T5:.2f}  (A0예산 {budget_all*100:.2f}%, n={len(allrows)})")
print(f"저장: {OUT_DIR / 'b2_scoretable_7item_raw.csv'} / {OUT_DIR / 'b2_scoretable_7item_rule.csv'}")


## 6. 검산 — 기존 산출값과 비교

sjlee님이 이미 만든 CSV 값과 이번 재현 결과가 일치하는지 확인 (일치하면 데이터 병합 로직이
`_recon_from_raw`와 동등하다는 뜻).

In [ ]:
expected_T4, expected_T5, expected_budget, expected_n = 0.08, 0.56, 17.81, 2196

print(f"기존값:  τ4={expected_T4}  τ5={expected_T5}  budget={expected_budget}%  n={expected_n}")
print(f"재현값:  τ4={T4}  τ5={T5}  budget={round(budget_all*100,2)}%  n={len(allrows)}")

if (T4, T5, len(allrows)) == (expected_T4, expected_T5, expected_n):
    print("\n✅ 완전 일치 — 데이터 병합 로직 검증됨.")
else:
    print("\n⚠️ 불일치 — adl_wide.csv/baseline_sample.csv 병합 방식이 _recon_from_raw와 다를 수 있음.")
    print("   sjlee님께 _recon_from_raw.py 로직 확인 요청 권장.")


## 6. 검산 — 기존 산출값과 비교

sjlee님이 이미 만든 CSV 값과 이번 재현 결과가 일치하는지 확인 (일치하면 데이터 병합 로직이
`_recon_from_raw`와 동등하다는 뜻).

In [ ]:
expected_T4, expected_T5, expected_budget, expected_n = 0.08, 0.56, 17.81, 2196

print(f"기존값:  τ4={expected_T4}  τ5={expected_T5}  budget={expected_budget}%  n={expected_n}")
print(f"재현값:  τ4={T4}  τ5={T5}  budget={round(budget_all*100,2)}%  n={len(allrows)}")

if (T4, T5, len(allrows)) == (expected_T4, expected_T5, expected_n):
    print("\n✅ 완전 일치 — 데이터 병합 로직 검증됨.")
else:
    print("\n⚠️ 불일치 — adl_wide.csv/baseline_sample.csv 병합 방식이 _recon_from_raw와 다를 수 있음.")
    print("   sjlee님께 _recon_from_raw.py 로직 확인 요청 권장.")


---
# Aim2 7문항 사후분석

⚠️ **먼저 처리할 것**: 위 5·6번 셀 실행 결과에서 기존값(τ4=0.08, n=2196, budget=17.81%)과
재현값(τ4=0.16, n=1358, budget=47.78%)이 크게 어긋남 — n이 838명(38%) 빠지고 A0예산이
거의 3배가 됨. 이 상태로 아래 분석을 이어가면 잘못된 데이터 기반 결론이 될 위험이 크므로,
7번(진단)부터 먼저 실행해서 원인을 찾을 것.

## 7. 데이터 정합성 진단 — n이 838명 빠진 원인 규명

In [ ]:
# ============================================================
# 7-1. com(baseline_sample 기준) ↔ adl_main(문항점수) 매칭 실패 지점 확인
# ============================================================
print(f"com(in_common_comparison_sample==True): {len(com)}명")
print(f"adl_main VISITNUM==2.0 전체: {(adl_main['VISITNUM']==2.0).sum()}명")

# USUBJID 완전일치 여부
com_ids = set(com["USUBJID"])
adl_base_ids = set(adl_main[adl_main["VISITNUM"]==2.0]["USUBJID"])
print(f"\ncom에는 있는데 adl_main(VISITNUM==2.0)에는 없는 USUBJID: {len(com_ids - adl_base_ids)}명")
print(f"adl_main에는 있는데 com에는 없는 USUBJID: {len(adl_base_ids - com_ids)}명")
print(f"양쪽 다 있는 USUBJID: {len(com_ids & adl_base_ids)}명")

# 문항별 결측률 (merge 후 full 기준) — 어떤 문항에서 특히 많이 빠지는지
print("\n=== full(merge 후) 문항별 결측 인원 ===")
for it in ALL_ITEMS_NEEDED:
    n_missing = full[it].isna().sum()
    print(f"  {it}({LAB.get(it,'?')}): 결측 {n_missing}명 / 전체 {len(full)}명")

# adl_main에 VISITNUM==2.0 중복 여부(중복이면 merge시 행 폭증 → 다른 문제로 이어질 수 있음)
dup_check = adl_main[adl_main["VISITNUM"]==2.0].groupby("USUBJID").size()
print(f"\nadl_main VISITNUM==2.0에서 USUBJID 중복(2개 이상): {(dup_check>1).sum()}명")


In [ ]:
# ============================================================
# 7-2. com에는 있는데 adl_main baseline에 없는 사람들 — 어디 갔는지 추적
# ============================================================
missing_from_adl = com_ids - adl_base_ids
if missing_from_adl:
    sample_missing = list(missing_from_adl)[:10]
    print(f"매칭 실패 샘플 10명: {sample_missing}")

    # 이 사람들이 adl_main에 다른 VISITNUM으로는 존재하는지
    other_visits = adl_main[adl_main["USUBJID"].isin(missing_from_adl)]
    print(f"\n이 {len(missing_from_adl)}명 중 adl_main에 (다른 VISITNUM으로라도) 존재하는 사람: "
          f"{other_visits['USUBJID'].nunique()}명")
    print(other_visits["VISITNUM"].value_counts())

    n_completely_absent = len(missing_from_adl) - other_visits["USUBJID"].nunique()
    print(f"\nadl_main 파일 자체에 전혀 없는 사람: {n_completely_absent}명 "
          f"(→ 이 경우 adl_wide.csv 자체가 이 사람들을 애초에 안 담고 있다는 뜻)")
else:
    print("USUBJID 매칭은 100% 성공 — 결측은 다른 원인(개별 문항 결측)일 가능성. 7-1 결과 재확인.")

print("\n※ 원인이 규명되면: (a) USUBJID 포맷 차이(공백/대소문자 등) → 정규화 후 재병합,")
print("   (b) in_common_comparison_sample 정의 자체가 adl_wide.csv 커버리지와 다름 → ")
print("      sjlee님 _recon_from_raw.py 로직 확인 필요, (c) 실제 결측 → 그대로 반영.")


## 8. 시험별(1061/1063/1064) 층화 다변량분석 — p-value 일치성 확인

이전 6문항 층화분석과 동일한 방식: **7문항을 동시투입한 다변량 로지스틱**을 시험별로 따로 적합하고,
Wald p-value가 시험 간 일치하는지, 완전분리 의심(Wald↔LR 불일치)이 있는지 확인.

⚠️ 7번 섹션에서 데이터 정합성 문제가 해결된 뒤 실행할 것(원인 규명 전까지는 잠정 결과로 취급).

In [ ]:
# ============================================================
# 8-1. 시험별 개별 다변량모형 적합 (절단별 P(Y>=k), k=1..5)
# ============================================================
import statsmodels.api as sm

FINAL7 = CAND[PRIMARY]  # 고정6 + Q7전화

def fit_cut_safe(df_, items, k):
    """완전분리 등으로 수렴 실패 시 예외 대신 표시만."""
    sub = df_.dropna(subset=items + ["ds_stage"]).copy()
    y = (sub["ds_stage"] >= k).astype(int)
    X = sm.add_constant(sub[items].astype(float))
    n_pos, n_neg = int(y.sum()), int((1 - y).sum())
    try:
        model = sm.Logit(y, X).fit(disp=0, maxiter=200)
        converged = model.mle_retvals.get("converged", True)
        wald_p = model.pvalues
        # LR test: 각 문항 하나씩 빼고 비교
        lr_p = {}
        for it in items:
            X_reduced = X.drop(columns=[it])
            try:
                m_reduced = sm.Logit(y, X_reduced).fit(disp=0, maxiter=200)
                lr_stat = 2 * (model.llf - m_reduced.llf)
                lr_p[it] = 1 - stats.chi2.cdf(lr_stat, df=1)
            except Exception:
                lr_p[it] = np.nan
        return {"converged": converged, "n_pos": n_pos, "n_neg": n_neg,
                "wald_p": wald_p, "lr_p": lr_p}
    except Exception as e:
        return {"converged": False, "n_pos": n_pos, "n_neg": n_neg,
                "wald_p": None, "lr_p": None, "error": str(e)}

from scipy import stats

stratified_rows = []
for trial in TRIALS:
    sub_trial = full[full["STUDYID"] == trial]
    print(f"===== {trial} (n={len(sub_trial)}) =====")
    for k in range(1, 6):
        r = fit_cut_safe(sub_trial, FINAL7, k)
        print(f"  P(Y>={k}): n_pos={r['n_pos']}, n_neg={r['n_neg']}, 수렴={r['converged']}")
        if r["wald_p"] is not None:
            for it in FINAL7:
                wald_p = r["wald_p"].get(it, np.nan)
                lr_p = r["lr_p"].get(it, np.nan)
                possible_sep = (wald_p > 0.5 and lr_p < 0.1) if pd.notna(wald_p) and pd.notna(lr_p) else False
                stratified_rows.append(dict(STUDYID=trial, cutpoint=f"P(Y>={k})", item=it,
                                             item_label=LAB.get(it, it),
                                             p_value=round(wald_p, 4) if pd.notna(wald_p) else np.nan,
                                             lr_p_value=round(lr_p, 4) if pd.notna(lr_p) else np.nan,
                                             possible_separation=possible_sep))

strat_df = pd.DataFrame(stratified_rows)
strat_df.to_csv(OUT_DIR / "b2_7item_stratified_coef_table.csv", index=False, encoding="utf-8-sig")
print(f"\n저장: {OUT_DIR / 'b2_7item_stratified_coef_table.csv'}")


In [ ]:
# ============================================================
# 8-2. 시험 간 유의성 일치 여부 비교표
# ============================================================
pivot = strat_df.pivot_table(index=["cutpoint", "item_label"], columns="STUDYID", values="p_value")
pivot.columns = [f"{c}_p" for c in pivot.columns]

sig = strat_df.pivot_table(index=["cutpoint", "item_label"], columns="STUDYID",
                            values="p_value", aggfunc=lambda x: (x < 0.05).any())
sig.columns = [f"{c}_유의" for c in sig.columns]

sep = strat_df.pivot_table(index=["cutpoint", "item_label"], columns="STUDYID",
                            values="possible_separation", aggfunc="any")
sep.columns = [f"{c}_분리의심" for c in sep.columns]

combined = pd.concat([pivot, sig, sep], axis=1)
n_sig_cols = [c for c in combined.columns if c.endswith("_유의")]
combined["n_significant"] = combined[n_sig_cols].sum(axis=1)
combined["시험간_불일치"] = (combined["n_significant"] > 0) & (combined["n_significant"] < 3)

print(f"시험 간 유의성 판정이 갈리는 행: {combined['시험간_불일치'].sum()} / {len(combined)}")
print("\n=== 불일치 행 ===")
print(combined[combined["시험간_불일치"]].round(4).to_string())

combined.to_csv(OUT_DIR / "b2_7item_stratified_cross_trial_comparison.csv", encoding="utf-8-sig")
print(f"\n저장: {OUT_DIR / 'b2_7item_stratified_cross_trial_comparison.csv'}")


### 8-3. 전체 28행 — 일치(16건)까지 포함한 전체 목록

In [ ]:
# ============================================================
# 8-3. 전체 28행(P(Y>=2)~P(Y>=5) × 7문항) 출력 — 일치/불일치 모두
# ============================================================
pd.set_option("display.max_rows", 40)
pd.set_option("display.width", 200)

print(f"전체 {len(combined)}행 (일치 {(~combined['시험간_불일치']).sum()}건 + "
      f"불일치 {combined['시험간_불일치'].sum()}건)\n")

print("=== 3개 시험 다 유의(n_significant==3) ===")
all_sig = combined[combined["n_significant"] == 3]
print(f"{len(all_sig)}건")
print(all_sig.round(4).to_string())

print("\n=== 3개 시험 다 비유의(n_significant==0) ===")
all_nonsig = combined[combined["n_significant"] == 0]
print(f"{len(all_nonsig)}건")
print(all_nonsig.round(4).to_string())

print("\n=== 전체 28행 (정렬: cutpoint, item_label) ===")
print(combined.round(4).to_string())

combined.round(4).to_csv(OUT_DIR / "b2_7item_stratified_full28.csv", encoding="utf-8-sig")
print(f"\n저장: {OUT_DIR / 'b2_7item_stratified_full28.csv'}")


### 해석 가이드

- 이전 6문항 층화분석에서 확인했던 패턴(P(Y≥1)은 완전분리로 사실상 무효, AD-1061이 P≥3·P≥4에서
  반복적으로 "예외"로 튀는 경향)이 **7문항(Q7전화 추가)에서도 재현되는지** 확인.
- `possible_separation=True`인 셀은 `p_value`(Wald) 대신 `lr_p_value`로 판단할 것 — Wald가
  완전분리로 가짜 비유의를 낼 수 있음.
- Q7전화가 새로 추가된 문항이므로, **Q7전화 행이 시험 간 일치하는지가 이번 분석의 핵심 체크포인트**.

## 9~11. 이후 이어갈 섹션 (개요만, 다음 단계에서 코드 작성)

- **9. 안정성 선택**: 훈련풀(dev+test 통합 또는 dev만) 부트스트랩 200회 재표집 → 7문항 각각
  |표준화계수|>0.25 비율 계산. Q7전화가 fold/재표집마다 안정적으로 뽑히는지가 핵심.
- **10. BIC/유효자유도**: `aim3_v2_wk12wk24_skeleton.py`의 `step6_bic_effective_df` 함수를
  훈련풀 기준으로 재사용 — 7문항이 CV 없이도 합리적 복잡도인지 사후확인.
- **11. l1_ratio 스윕 + DCA**: 우선순위 낮음, 위 항목들 정리 후 착수.

## 9~11. Aim2 최종 확정 근거 보강 (7문항)

데이터 버그 해결 확인됐으므로 `full`(dev+test 결합, n=2,203) 기준으로 진행.
이 세 섹션은 **문항 재선택이 아니라 이미 확정한 7문항의 견고성을 사후확인**하는 목적이므로,
test를 추가로 다시 여는 것과는 성격이 다름(새로운 후보를 test로 고르지 않음).

## 9. 안정성 선택 (부트스트랩 200회) — Q7전화 안정성 핵심 체크

In [ ]:
# ============================================================
# 9-1. 부트스트랩 안정성 선택
# ============================================================
def bootstrap_stability_selection(df_, items, n_boot=200, coef_thresh=0.25, seed=0):
    """각 문항이 재표집마다 |표준화계수|>coef_thresh인 절단이 하나라도 있는 비율(=안정성 지수)."""
    rng = np.random.RandomState(seed)
    d = df_.dropna(subset=items + ["ds_stage"]).reset_index(drop=True)
    n = len(d)
    selected_counts = {it: 0 for it in items}

    for b in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        boot = d.iloc[idx]
        X = boot[items].values.astype(float)
        y = boot["ds_stage"].values
        sc = StandardScaler().fit(X)
        Z = sc.transform(X)

        max_abs_coef = np.zeros(len(items))
        for k in range(1, 6):
            yk = (y >= k).astype(int)
            if len(np.unique(yk)) < 2:
                continue
            m = LogisticRegression(penalty="l2", solver="lbfgs", C=1.0, max_iter=2000).fit(Z, yk)
            max_abs_coef = np.maximum(max_abs_coef, np.abs(m.coef_[0]))

        for j, it in enumerate(items):
            if max_abs_coef[j] > coef_thresh:
                selected_counts[it] += 1

    return {it: round(selected_counts[it] / n_boot, 3) for it in items}

print("부트스트랩 200회 진행 중... (몇 분 소요될 수 있음)")
stability = bootstrap_stability_selection(full, FINAL7, n_boot=200, coef_thresh=0.25, seed=0)

stab_df = pd.DataFrame([{"문항": LAB[it], "item_code": it, "안정성지수": v} for it, v in stability.items()])
stab_df = stab_df.sort_values("안정성지수", ascending=False)
print(stab_df.to_string(index=False))

stab_df.to_csv(OUT_DIR / "b2_7item_stability_selection.csv", index=False, encoding="utf-8-sig")
print(f"\n저장: {OUT_DIR / 'b2_7item_stability_selection.csv'}")
print("\n※ 안정성지수 0.8 이상이면 fold/재표집에 안 흔들리는 견고한 문항.")
print("   Q7전화가 고정6 문항들과 비슷한 수준으로 나오는지가 핵심 체크포인트.")


## 10. BIC / 유효자유도 — 7문항 복잡도 사후검증

In [ ]:
# ============================================================
# 10-1. BIC/유효자유도 계산 (릿지, SVD 기반)
# ============================================================
from sklearn.linear_model import Ridge

def bic_effective_df(df_, items, target_col="ds_stage", lam=1.0):
    d = df_.dropna(subset=items + [target_col])
    X = d[items].values.astype(float)
    y = d[target_col].values.astype(float)
    n = len(y)

    Xc = X - X.mean(axis=0)
    yc = y - y.mean()
    _, s, _ = np.linalg.svd(Xc, full_matrices=False)
    eff_df = float(np.sum(s**2 / (s**2 + lam)))

    ridge = Ridge(alpha=lam).fit(Xc, yc)
    resid = yc - ridge.predict(Xc)
    rss = float(np.sum(resid**2))
    bic = n * np.log(rss / n) + np.log(n) * eff_df

    return {"n": n, "문항수": len(items), "유효자유도": round(eff_df, 2),
            "RSS": round(rss, 2), "BIC": round(bic, 2)}

# CAND(7~9문항 후보 전부) + 고정6(문항 최소 기준선)까지 나란히 비교
bic_targets = {"6문항(고정만)": FIXED6, **CAND}

bic_rows = []
for label, items in bic_targets.items():
    r = bic_effective_df(full, items)
    r["조합"] = label
    bic_rows.append(r)

bic_df = pd.DataFrame(bic_rows).set_index("조합")[["문항수", "n", "유효자유도", "RSS", "BIC"]]
bic_df = bic_df.sort_values("BIC")
print(bic_df.to_string())
print(f"\n★ 최종 확정({PRIMARY})의 BIC 순위: "
      f"{list(bic_df.index).index(PRIMARY) + 1}위 / {len(bic_df)}개 중")

bic_df.to_csv(OUT_DIR / "b2_7item_bic_comparison.csv", encoding="utf-8-sig")
print(f"저장: {OUT_DIR / 'b2_7item_bic_comparison.csv'}")
print("\n※ CV 없이(held-out 불필요) 훈련데이터 1회 적합만으로 얻은 독립적 근거.")
print("   7문항이 최하위권 BIC가 아니면 '과적합 아니다'는 근거로 인용 가능.")


## 11. l1_ratio 스윕 + DCA

In [ ]:
# ============================================================
# 11-1. l1_ratio 스윕 (dev 내부 5-fold CV, test는 안 건드림)
# ============================================================
from sklearn.model_selection import StratifiedKFold

def l1ratio_sweep_cv(df_, items, l1_grid=np.linspace(0, 1, 11), n_splits=5, seed=0):
    d = df_.dropna(subset=items + ["ds_stage"]).reset_index(drop=True)
    X_all = d[items].values.astype(float)
    y_all = d["ds_stage"].values
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    rows = []
    for l1 in l1_grid:
        fold_maes = []
        for tr_idx, va_idx in skf.split(X_all, y_all):
            Xtr, Xva = X_all[tr_idx], X_all[va_idx]
            ytr, yva = y_all[tr_idx], y_all[va_idx]
            sc = StandardScaler().fit(Xtr)
            Ztr, Zva = sc.transform(Xtr), sc.transform(Xva)

            penalty = "l2" if l1 == 0 else "elasticnet"
            kwargs = dict(penalty=penalty, C=1.0, max_iter=3000, random_state=0)
            if penalty == "elasticnet":
                kwargs.update(solver="saga", l1_ratio=l1)
            else:
                kwargs.update(solver="lbfgs")

            g = {}
            for k in range(1, 6):
                yk = (ytr >= k).astype(int)
                if len(np.unique(yk)) < 2:
                    g[k] = None
                else:
                    g[k] = LogisticRegression(**kwargs).fit(Ztr, yk)

            P = np.zeros((len(Zva), 6))
            probs1_5 = {k: (g[k].predict_proba(Zva)[:, 1] if g[k] is not None else np.zeros(len(Zva)))
                        for k in range(1, 6)}
            P[:, 0] = 1 - probs1_5[1]
            for k in range(1, 5):
                P[:, k] = probs1_5[k] - probs1_5[k + 1]
            P[:, 5] = probs1_5[5]
            P = np.clip(P, 1e-9, None); P /= P.sum(1, keepdims=True)
            pred = cmed(P)  # 단순 조건부중앙값 규칙(스윕은 근거자료용이라 τ 튜닝 생략)
            fold_maes.append(mae(yva, pred))
        rows.append({"l1_ratio": round(l1, 2), "CV_MAE_평균": round(np.mean(fold_maes), 4)})
    return pd.DataFrame(rows)

sweep_df = l1ratio_sweep_cv(dev, FINAL7)
print(sweep_df.to_string(index=False))
print(f"\n최소 MAE: l1_ratio={sweep_df.loc[sweep_df['CV_MAE_평균'].idxmin(),'l1_ratio']} "
      f"(0=순수L2, 1=순수라쏘)")
sweep_df.to_csv(OUT_DIR / "b2_7item_l1ratio_sweep.csv", index=False, encoding="utf-8-sig")


In [ ]:
# ============================================================
# 11-2. DCA (결정곡선분석) — 최종모형(7문항) 순이득 곡선
# ============================================================
def decision_curve_analysis(y_true, y_pred_prob, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.01, 0.51, 0.01)
    y_true = np.asarray(y_true, dtype=float)
    n = len(y_true)
    event_rate = y_true.mean()

    rows = []
    for pt in thresholds:
        pred_pos = (y_pred_prob >= pt).astype(float)
        tp = np.sum((pred_pos == 1) & (y_true == 1))
        fp = np.sum((pred_pos == 1) & (y_true == 0))
        nb_model = tp / n - fp / n * (pt / (1 - pt))
        nb_all = event_rate - (1 - event_rate) * (pt / (1 - pt))
        rows.append({"threshold": pt, "net_benefit_model": nb_model,
                     "net_benefit_treat_all": nb_all, "net_benefit_treat_none": 0.0})
    return pd.DataFrame(rows)

# 최종 7문항 모형(dev로 적합, test로 확률 산출) — 중증(ds_stage>=4) 판별 DCA
dtr = dev.dropna(subset=FINAL7); dte = test.dropna(subset=FINAL7)
sc, md_final = fit_std(dtr[FINAL7].values, dtr["ds_stage"].values)
Pte_final = P_(sc, md_final, dte[FINAL7].values)
p4_prob = Pte_final[:, 4] + Pte_final[:, 5]  # P(중증) = P(Y=4)+P(Y=5)
y_severe = (dte["ds_stage"] >= 4).astype(int).values

dca = decision_curve_analysis(y_severe, p4_prob)
print(dca.head(10).to_string(index=False))
dca.to_csv(OUT_DIR / "b2_7item_dca.csv", index=False, encoding="utf-8-sig")
print(f"\n저장: {OUT_DIR / 'b2_7item_dca.csv'}")

print("\nColab에서 시각화:")
print("  import matplotlib.pyplot as plt")
print("  plt.plot(dca.threshold, dca.net_benefit_model, label='7문항 모형')")
print("  plt.plot(dca.threshold, dca.net_benefit_treat_all, '--', label='treat-all')")
print("  plt.axhline(0, color='gray', ls=':', label='treat-none')")
print("  plt.xlabel('위험임계값'); plt.ylabel('순이득'); plt.legend(); plt.show()")


In [ ]:
!pip install koreanize-matplotlib -q
import koreanize_matplotlib  # import만 해도 자동 적용됨

In [ ]:
import matplotlib.pyplot as plt
plt.plot(dca.threshold, dca.net_benefit_model, label='7문항 모형')
plt.plot(dca.threshold, dca.net_benefit_treat_all, '--', label='treat-all')
plt.axhline(0, color='gray', ls=':', label='treat-none')
plt.xlabel('위험임계값'); plt.ylabel('순이득'); plt.legend(); plt.show()

## 12. 최종 매핑표 — 문항코드/라벨 + 원눈금 계수 + 유의성 통합

⚠️ 계수(릿지, C=1.0)와 유의성(p-value)은 원래 서로 다른 모형·표본에서 나왔던 값이라 그대로
붙이면 정합성이 깨짐. **여기서는 유의성을 채점표와 동일한 표본(full, n=2,196 전체)에서
벌점없는 로지스틱으로 새로 계산**해서 계수표와 병합함 — 8절(시험별 층화)과는 다른 목적(전체
표본 기준 유의성)이니 혼동하지 말 것.

In [ ]:
# ============================================================
# 12-1. 전체 표본(full, n=2,196) 기준 유의성(p-value) 계산
# ============================================================
allrows_final = full.dropna(subset=FINAL7 + ["ds_stage"]).reset_index(drop=True)
X_final = allrows_final[FINAL7].astype(float)
y_final = allrows_final["ds_stage"].values

sig_rows = []
for k in range(1, 6):
    yk = (y_final >= k).astype(int)
    Xc = sm.add_constant(X_final)

    m = None
    converged = False
    try:
        m = sm.Logit(yk, Xc).fit(disp=0, maxiter=200)
        converged = m.mle_retvals.get("converged", True)
    except np.linalg.LinAlgError:
        try:
            # 뉴턴법이 특이행렬로 실패하면 bfgs로 재시도 (행렬역산 없이 수렴 가능)
            m = sm.Logit(yk, Xc).fit(disp=0, maxiter=500, method="bfgs")
            converged = m.mle_retvals.get("converged", True)
        except Exception:
            m = None
            converged = False

    if m is None:
        for it in FINAL7:
            sig_rows.append(dict(cutpoint=f"P(Y>={k})", item_code=it, item_label=LAB[it],
                                  p_value=np.nan, significant=False, converged=False))
        print(f"⚠️ P(Y>={k}) 완전분리로 추정 자체 불가 — 이 절단은 전부 NaN 처리")
        continue

    for it in FINAL7:
        p = m.pvalues.get(it, np.nan)
        sig_rows.append(dict(cutpoint=f"P(Y>={k})", item_code=it, item_label=LAB[it],
                              p_value=round(p, 4) if pd.notna(p) else np.nan,
                              significant=bool(p < 0.05) if pd.notna(p) else False,
                              converged=converged))
    if not converged:
        print(f"⚠️ P(Y>={k}) 전체표본 수렴 실패 — 이 절단 유의성은 참고만")

sig_df = pd.DataFrame(sig_rows)
print(sig_df.to_string(index=False))


In [ ]:
# ============================================================
# 12-2. 채점표(원눈금 계수, 5번 섹션 'score') + 유의성 병합 → 최종 매핑표
# ============================================================
mapping_table = score.merge(sig_df, on=["cutpoint", "item_code", "item_label"], how="left")

def stars(p):
    if pd.isna(p): return ""
    if p < 0.001: return "***"
    if p < 0.01: return "**"
    if p < 0.05: return "*"
    return ""

mapping_table["sig_stars"] = mapping_table["p_value"].apply(stars)

# 절편 행은 유의성 대상 아님 — 표시만 깔끔하게
mapping_table.loc[mapping_table["item_code"] == "(intercept)",
                   ["p_value", "significant", "sig_stars", "converged"]] = [np.nan, False, "", np.nan]

cols_order = ["cutpoint", "item_code", "item_label", "ridge_coef_raw",
              "ridge_coef_standardized", "p_value", "sig_stars", "significant", "converged"]
mapping_table = mapping_table[cols_order]

print(mapping_table.to_string(index=False))

mapping_table.to_csv(OUT_DIR / "b2_7item_final_mapping_with_significance.csv",
                      index=False, encoding="utf-8-sig")
print(f"\n저장: {OUT_DIR / 'b2_7item_final_mapping_with_significance.csv'}")


### 참고

- `ridge_coef_*`는 실제 채점에 쓰는 값(릿지, 전체표본 재학습) 그대로.
- `p_value`/`sig_stars`는 **같은 표본**에서 벌점 없는 모형으로 따로 구한 유의성 — 계수 크기와
  유의성이 항상 같은 방향은 아닐 수 있음(릿지는 계수를 축소시키므로, 유의하지만 릿지계수가
  작게 눌린 문항이 있을 수 있음 — 이 경우 값 자체보다 '방향과 유의성'을 보고할 것).
- P(Y≥1)은 8절에서 이미 확인했듯 극단적 불균형으로 수렴 실패 가능성 높음 — `converged`
  컬럼으로 확인하고, False면 그 절단의 p_value는 참고만 할 것.